# Majority-class baseline

This notebook evaluates a majority-class baseline using the same loading, chronological split, and preprocessing pipeline as later models. The data is divided into train (70%), validation (15%), and test (15%) partitions in planned-time order.

The baseline is fitted only on the training partition. Validation is reserved for model selection, and the test partition is used once for final evaluation. Macro-F1 is the primary metric because each delay class should contribute equally to the comparison.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from macro_f1_test import macro_f1_test
from prep_job import (
    add_split_time,
    add_target,
    chronological_split,
    load_monthly_data,
    preprocess_partitions,
)

## Load, split, and preprocess the data

The same preprocessing pipeline is used for the baseline and future models. Schedule information is prepared for chronological ordering, the partitions are created, and preprocessing is then applied independently to each partition.

In [ ]:
columns = [
    "arrival_planned_time",
    "departure_planned_time",
    "time",
    "delay_in_min",
    "arrival_is_canceled",
    "train_type",
    "line_number",
    "station_name",
    "train_number",
    "train_line_station_num",
    "final_destination_station",
]

data = load_monthly_data(PROJECT_ROOT / "data", columns)
labelled = add_target(data)
partitions = chronological_split(add_split_time(labelled))
processed = preprocess_partitions(partitions)

pd.DataFrame(
    {
        "partition": list(partitions),
        "rows_before_preprocessing": [len(partitions[name]) for name in partitions],
        "rows_after_preprocessing": [len(processed[name]) for name in processed],
    }
).assign(
    removed=lambda frame: frame["rows_before_preprocessing"]
    - frame["rows_after_preprocessing"]
)

## Majority baseline by train type

Each train type receives a separate `DummyClassifier`. The model predicts the most frequent target class found in its training partition. A constant feature is used because this baseline does not use predictive features.

In [ ]:
results = []

for train_type in ["IC", "ICE", "RE"]:
    train_group = processed["train"].query("train_type == @train_type")
    test_group = processed["test"].query("train_type == @train_type")

    if train_group.empty or test_group.empty:
        raise ValueError(f"Missing training or test rows for train type: {train_type}")

    model = DummyClassifier(strategy="most_frequent")
    model.fit(np.zeros((len(train_group), 1)), train_group["target"])
    result = macro_f1_test(
        model,
        np.zeros((len(test_group), 1)),
        test_group["target"],
        model_name=f"majority_{train_type}",
    )
    result["train_type"] = train_type
    result["training_majority_class"] = train_group["target"].value_counts().idxmax()
    results.append(result)

baseline_results = (
    pd.DataFrame(results)
    [["train_type", "model", "training_majority_class", "n_test", "macro_f1", "accuracy"]]
    .sort_values("train_type")
    .reset_index(drop=True)
)
baseline_results

## Summary

This result is the minimum reference point for later models. A model should be compared with the majority baseline using the same processed partitions and should improve macro-F1 without relying on the test data during training or tuning. Accuracy is reported as a secondary metric because it can be dominated by the most common delay class.